# Linha de base: agrupadores profundos do aeon no UCR (versão para supercomputador)

Treina os 6 agrupadores profundos do `aeon` 1.6.0 (AEFCN, AEResNet, AEDCNN, AEBiGRU,
AEAttentionBiGRU, AEDRNN) nos 128 conjuntos univariados do UCR e busca o número de grupos
**K de 2 a 70, de 2 em 2** (mais K = número de classes, para comparação).

**Protocolo** (Paparrizos e Bogireddy, 2025): treino e teste unidos, rótulos usados só nas
métricas; valores ausentes por interpolação linear; comprimentos diferentes reamostrados para o
maior (`Resizer("max")`); normalização z por série; 200 épocas; semente 0. Cada par
(conjunto, modelo) é treinado **uma vez**; o espaço latente (`model_.layers[1]`, o mesmo que o
aeon usa) é salvo, e o `TimeSeriesKMeans` euclidiano roda para cada K sobre ele.

**Como usar**
1. Rode a célula 1 (instalação) uma vez, se o ambiente ainda não tiver os pacotes.
2. Ajuste a célula 2 (configuração). Para conferir o ambiente, rode antes com `TESTE_RAPIDO = True`
   (3 conjuntos pequenos, 2 épocas, poucos minutos).
3. Rode o restante. Com `EM_SEGUNDO_PLANO = True`, o agendador continua rodando mesmo se o
   navegador ou o kernel forem fechados; use a célula de acompanhamento para ver o progresso.
4. **É retomável:** rodar de novo pula os pares já concluídos.

Cada par roda num subprocesso próprio. Se faltar memória ou travar, só aquele par falha e a
falha fica registrada em `execucoes.csv` / `falhas.csv`. Com várias GPUs, roda um par por GPU
em paralelo.

## 1. Instalação (uma vez)

In [ ]:
# Descomente se o ambiente não tiver os pacotes. Versões usadas na validação:
# aeon 1.6.0, tensorflow 2.21.0, keras 3.15.1, scikit-learn 1.9.1, numpy 2.3.5, pandas 3.0.6.
# Em Linux com GPU NVIDIA, "tensorflow[and-cuda]" traz CUDA/cuDNN pelo pip.
# %pip install -q "aeon==1.6.0" "tensorflow[and-cuda]==2.21.0" scikit-learn pandas

## 2. Configuração

In [ ]:
import os, json, subprocess, sys
from pathlib import Path

TESTE_RAPIDO = os.environ.get("TESTE_RAPIDO", "0") == "1"   # True: 3 conjuntos pequenos, 2 épocas

PASTA_TRABALHO = Path(os.environ.get("AEON_HPC_DIR", "aeon_dl_hpc")).resolve()
OUT      = PASTA_TRABALHO / ("resultados_teste" if TESTE_RAPIDO else "resultados")
DATA_DIR = PASTA_TRABALHO / "dados"   # baixa o UCR aqui; se o nó não tiver internet, copie para dados/ucr

EPOCHS  = 2 if TESTE_RAPIDO else 200
SEED    = 0
K_MIN, K_MAX, K_STEP = 2, 70, 2       # pedido do colaborador: 2:70:2
K_MODES = ["grid", "n_classes"]       # grade 2..70 e K = número de classes

MODELS = ["AEFCNClusterer", "AEResNetClusterer", "AEDCNNClusterer",
          "AEBiGRUClusterer", "AEAttentionBiGRUClusterer", "AEDRNNClusterer"]
MODELOS_CPU = [s for s in os.environ.get("MODELOS_CPU", "").split(",") if s]  # forçar CPU (ex.: GPU pequena)

DATASETS = None                       # None = os 128 univariados do UCR
if TESTE_RAPIDO:
    DATASETS = ["Coffee", "Beef", "GunPoint"]

def _gpus():
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
                           capture_output=True, text=True, timeout=30)
        return [l.strip() for l in r.stdout.splitlines() if l.strip()]
    except Exception:
        return []

GPUS = _gpus()                        # um par por GPU em paralelo; [] = CPU
JOBS_POR_GPU = 1                      # suba para 2 em GPUs grandes (>= 24 GB) se quiser
TIMEOUT_PAR_HORAS = 12                # par que passar disso é encerrado e marcado "timeout"
PARAR_APOS_HORAS = None               # ex.: 11 -> não inicia novos pares depois de 11 h
REPETIR_FALHAS = False                # True: tenta de novo os pares que falharam
EM_SEGUNDO_PLANO = not TESTE_RAPIDO   # True: agendador destacado do kernel (nohup)

print("GPUs:", GPUS or "nenhuma (CPU)")
print("saída:", OUT)

## 3. Código (gravado em arquivos `.py` ao lado do notebook)

In [ ]:
%%writefile aeon_dl_par.py
"""Treina UM par (conjunto, modelo) e grava tudo na pasta do par.

Chamado pelo agendador (aeon_dl_agendador.py), um subprocesso por par:
    python aeon_dl_par.py --config <out>/config.json --dataset Coffee --model AEFCNClusterer

Protocolo (CLAUDE.md, secao 4): treino+teste concatenados (rotulos so nas metricas),
NaN -> interpolacao linear, comprimentos diferentes -> Resizer("max"), z-norm por serie.
O autoencoder e treinado uma vez; o latente (model_.layers[1]) e salvo; para cada K roda
TimeSeriesKMeans euclidiano sobre o latente, igual ao aeon.

Saidas em <out>/pares/<modelo>/<conjunto>/:
    metricas.csv     formato longo (uma linha por K e metrica)
    curva_perda.csv  perda, taxa de aprendizado e tempo de cada epoca
    latente.npy      espaco latente (n_casos x dim)
    status.json      status final + informacoes de treino (escrito por ultimo)
"""

from __future__ import annotations

import argparse
import csv
import gc
import json
import os
import platform
import shutil
import socket
import tempfile
import time
import traceback
from datetime import datetime
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np  # noqa: E402

MODELS = [
    "AEFCNClusterer",
    "AEResNetClusterer",
    "AEDCNNClusterer",
    "AEBiGRUClusterer",
    "AEAttentionBiGRUClusterer",
    "AEDRNNClusterer",
]

LONG_COLS = ["dataset", "model", "k", "k_mode", "metric", "value", "seed", "n_epochs",
             "n_cases", "n_timepoints", "n_classes", "preprocessing"]

# Metricas gravadas para cada K (ver o dicionario no notebook).
METRICS = [
    "silhouette", "davies_bouldin", "calinski_harabasz",
    "ri", "ari", "nmi", "ami",
    "ri_train", "ari_train", "nmi_train", "ri_test", "ari_test", "nmi_test",
    "n_clusters_produced", "n_empty_components", "degenerate",
    "cluster_size_min", "cluster_size_max", "kmeans_inertia", "kmeans_n_iter",
    "recon_loss", "recon_mse", "elbo",
]


# ----------------------------------------------------------------------- dados
def _interp_nan(v: np.ndarray) -> tuple[np.ndarray, str]:
    mask = np.isnan(v)
    if not mask.any():
        return v, ""
    if mask.all():
        return np.zeros_like(v), "all_nan"
    idx = np.arange(v.size)
    v = v.copy()
    v[mask] = np.interp(idx[mask], idx[~mask], v[~mask])  # extremos: valor valido mais proximo
    return v, "nan"


def preprocess(series: list[np.ndarray]) -> tuple[np.ndarray, str]:
    """NaN -> interpolacao linear; comprimentos diferentes -> Resizer("max"); z-norm."""
    from aeon.transformations.collection.unequal_length import Resizer

    if series[0].ndim != 2 or series[0].shape[0] != 1:
        raise ValueError(f"formato inesperado: {series[0].shape}")
    n_nan = n_all_nan = 0
    clean = []
    for s in series:
        ch, flag = [], ""
        for c in range(s.shape[0]):
            v, f = _interp_nan(s[c])
            ch.append(v)
            flag = f or flag
        n_nan += flag == "nan"
        n_all_nan += flag == "all_nan"
        clean.append(np.stack(ch))
    lengths = np.array([s.shape[-1] for s in clean])
    steps = []
    if n_nan or n_all_nan:
        steps.append(f"interp_linear_nan(series={n_nan},all_nan_to_zero={n_all_nan})")
    if lengths.min() != lengths.max():
        X = Resizer(resized_length="max").fit_transform(clean)
        steps.append(f"resize_max(len_min={lengths.min()},len_max={lengths.max()})")
    else:
        X = np.stack(clean)
    X = np.asarray(X, dtype=np.float64)
    mu = X.mean(axis=2, keepdims=True)
    sd = X.std(axis=2, keepdims=True)
    sd[sd == 0] = 1.0
    X = ((X - mu) / sd).astype(np.float32)
    steps.append("znorm_per_series")
    return X, ";".join(steps)


def load_dataset(name: str, data_dir: Path) -> dict:
    """Carrega (com cache .npz), concatena treino+teste e pre-processa.

    O cache guarda n_train para permitir as metricas restritas ao split original.
    Gravacao atomica: varios processos podem carregar o mesmo conjunto ao mesmo tempo.
    """
    cache_dir = data_dir / "processed"
    cache = cache_dir / f"{name}.npz"
    if cache.exists():
        d = np.load(cache, allow_pickle=False)
        return {"X": d["X"], "y": d["y"], "n_train": int(d["n_train"]),
                "preprocessing": str(d["preprocessing"])}

    from aeon.datasets import load_classification

    raw = data_dir / "ucr"
    raw.mkdir(parents=True, exist_ok=True)
    Xtr, ytr = load_classification(name, split="train", extract_path=str(raw))
    Xte, yte = load_classification(name, split="test", extract_path=str(raw))
    series = [np.asarray(s, dtype=np.float64) for s in list(Xtr) + list(Xte)]
    y = np.concatenate([np.asarray(ytr), np.asarray(yte)]).astype(str)
    X, prep = preprocess(series)
    prep = "concat_train_test;" + prep
    cache_dir.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(suffix=".npz", dir=cache_dir)
    os.close(fd)
    np.savez(tmp, X=X, y=y, n_train=np.array(len(Xtr)), preprocessing=np.array(prep))
    os.replace(tmp, cache)
    return {"X": X, "y": y, "n_train": len(Xtr), "preprocessing": prep}


# ---------------------------------------------------------------------- modelo
def _callbacks():
    """Mesmo ReduceLROnPlateau do padrao do aeon + registro por epoca.

    Quando recebe callbacks, o aeon acrescenta o proprio ModelCheckpoint(monitor="loss",
    save_best_only), que recarrega o modelo de menor perda no fim, como no padrao.
    """
    import tensorflow as tf

    class EpochLog(tf.keras.callbacks.Callback):
        def on_train_begin(self, logs=None):
            self.rows = []

        def on_epoch_begin(self, epoch, logs=None):
            self._t = time.perf_counter()

        def on_epoch_end(self, epoch, logs=None):
            dt = time.perf_counter() - self._t
            try:
                lr = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))
            except Exception:  # noqa: BLE001
                lr = float("nan")
            self.rows.append({"epoch": epoch + 1, "loss": float((logs or {}).get("loss", np.nan)),
                              "learning_rate": lr, "epoch_time_s": dt})

    log = EpochLog()
    rlr = tf.keras.callbacks.ReduceLROnPlateau(monitor="loss", factor=0.5, patience=50,
                                               min_lr=0.0001)
    return [rlr, log], log


def _check_encoder(clst, X: np.ndarray, latent: np.ndarray) -> str:
    """Confere que model_.layers[1] e o codificador (mesma checagem da linha de base)."""
    Xk = X.transpose(0, 2, 1)
    layers = clst.model_.layers
    sub = Xk[: min(256, len(Xk))]
    full = clst.model_.predict(sub, verbose=0)
    comp = layers[2].predict(layers[1].predict(sub, verbose=0), verbose=0)
    recon_ok = bool(np.allclose(full, comp, atol=1e-5))
    pred_ok = bool(np.array_equal(clst.predict(X), clst.estimator_.predict(latent)))
    ok = len(layers) == 3 and recon_ok and pred_ok
    return (f"{'OK' if ok else 'FALHOU'} layers={[ly.name for ly in layers]} "
            f"dec(enc(x))==model(x):{recon_ok} predict_igual:{pred_ok}")


def train_and_extract(model_name: str, X: np.ndarray, n_epochs: int, seed: int):
    import aeon.clustering.deep_learning as dl
    import tensorflow as tf

    shm = "/dev/shm" if os.path.isdir("/dev/shm") and os.access("/dev/shm", os.W_OK) else None
    tmp = tempfile.mkdtemp(prefix="aeon_ckpt_", dir=shm)
    try:
        cbs, log = _callbacks()
        clst = getattr(dl, model_name)(
            callbacks=cbs, n_epochs=n_epochs, random_state=seed, batch_size=32,
            save_best_model=False, save_last_model=False,
            file_path=tmp.rstrip("/\\") + "/", verbose=False)
        t0 = time.perf_counter()
        clst.fit(X)
        train_t = time.perf_counter() - t0

        hist = clst.history.history if hasattr(clst.history, "history") else clst.history
        losses = [float(v) for v in hist["loss"]]
        Xk = X.transpose(0, 2, 1)
        t0 = time.perf_counter()
        latent = clst.model_.layers[1].predict(Xk, verbose=0)
        latent = np.asarray(latent).reshape(len(X), -1).astype(np.float32)
        latent_t = time.perf_counter() - t0
        check = _check_encoder(clst, X, latent)
        recon_mse = float(np.mean((clst.model_.predict(Xk, verbose=0) - Xk) ** 2))

        mem_peak = ""
        if tf.config.list_physical_devices("GPU"):
            try:
                mem_peak = round(tf.config.experimental.get_memory_info("GPU:0")["peak"] / 2**20, 1)
            except Exception:  # noqa: BLE001
                pass
        times = [r["epoch_time_s"] for r in log.rows]
        info = {
            "n_params": int(clst.model_.count_params()),
            "latent_dim": int(latent.shape[1]),
            "train_time_s": round(train_t, 3),
            "latent_time_s": round(latent_t, 3),
            "epochs_run": len(losses),
            "best_epoch": int(np.argmin(losses)) + 1,
            "first_epoch_s": round(times[0], 3) if times else "",
            "epoch_median_s": round(float(np.median(times[1:])), 4) if len(times) > 1 else "",
            "recon_loss_min": min(losses),
            "recon_loss_last": losses[-1],
            "recon_mse": recon_mse,
            "lr_final": log.rows[-1]["learning_rate"] if log.rows else "",
            "gpu_mem_peak_mb": mem_peak,
            "encoder_check": check,
        }
        return latent, info, log.rows
    finally:
        tf.keras.backend.clear_session()
        gc.collect()
        shutil.rmtree(tmp, ignore_errors=True)


# -------------------------------------------------------------------- busca K
def _external(y, lab, M) -> tuple[float, float, float]:
    return (M.rand_score(y, lab), M.adjusted_rand_score(y, lab),
            M.normalized_mutual_info_score(y, lab))


def k_search(latent, y, n_train, ks, k_mode, seed, base, recon_loss, recon_mse):
    from aeon.clustering import TimeSeriesKMeans
    from sklearn import metrics as M

    rows, notes = [], []

    def emit(k, vals):
        for m in METRICS:
            v = vals.get(m, float("nan"))
            rows.append({**base, "k": k, "k_mode": k_mode, "metric": m,
                         "value": "" if v == "" else repr(float(v))})

    for k in ks:
        km = TimeSeriesKMeans(n_clusters=k, distance="euclidean", averaging_method="mean",
                              random_state=seed)
        try:
            lab = np.asarray(km.fit_predict(latent))
        except Exception as e:  # noqa: BLE001  (ex.: latente degenerado)
            notes.append(f"K={k}: k-means falhou ({type(e).__name__}: {e}); metricas=NaN")
            emit(k, {"recon_loss": recon_loss, "recon_mse": recon_mse, "elbo": ""})
            continue
        sizes = np.bincount(np.unique(lab, return_inverse=True)[1])
        n_found = len(sizes)
        vals = {"n_clusters_produced": n_found, "n_empty_components": k - n_found,
                "degenerate": float(n_found < 2),
                "cluster_size_min": sizes.min(), "cluster_size_max": sizes.max(),
                "kmeans_inertia": getattr(km, "inertia_", float("nan")),
                "kmeans_n_iter": getattr(km, "n_iter_", float("nan"))}
        if 2 <= n_found <= len(lab) - 1:
            vals["silhouette"] = M.silhouette_score(latent, lab, metric="euclidean")
            vals["davies_bouldin"] = M.davies_bouldin_score(latent, lab)
            vals["calinski_harabasz"] = M.calinski_harabasz_score(latent, lab)
        else:
            notes.append(f"K={k}: {n_found} grupo(s) distinto(s); silhueta/DB/CH=NaN")
        if n_found < k:
            notes.append(f"K={k}: k-means devolveu {n_found} grupos")
        vals["ri"], vals["ari"], vals["nmi"] = _external(y, lab, M)
        vals["ami"] = M.adjusted_mutual_info_score(y, lab)
        # Mesmo agrupamento (ajustado no conjunto unido), avaliado so nas series que vieram
        # do treino e so nas que vieram do teste no split original do UCR.
        if 0 < n_train < len(y):
            vals["ri_train"], vals["ari_train"], vals["nmi_train"] = _external(
                y[:n_train], lab[:n_train], M)
            vals["ri_test"], vals["ari_test"], vals["nmi_test"] = _external(
                y[n_train:], lab[n_train:], M)
        vals.update(recon_loss=recon_loss, recon_mse=recon_mse, elbo="")
        emit(k, vals)
    return rows, notes


def ks_for(k_mode, n_cases, n_classes, cfg):
    if k_mode == "grid":
        return [k for k in range(cfg["k_min"], cfg["k_max"] + 1, cfg["k_step"]) if k < n_cases]
    return [n_classes]


# ----------------------------------------------------------------------- main
def pair_dir(out: Path, dataset: str, model: str) -> Path:
    return out / "pares" / model / dataset


def _write_csv(path: Path, cols, rows):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        for r in rows:
            w.writerow({c: r.get(c, "") for c in cols})


def _write_status(d: Path, status: dict):
    tmp = d / "status.json.tmp"
    tmp.write_text(json.dumps(status, indent=1, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, d / "status.json")


def run_pair(cfg: dict, dataset: str, model: str) -> int:
    out = Path(cfg["out"])
    d = pair_dir(out, dataset, model)
    d.mkdir(parents=True, exist_ok=True)
    for f in ("status.json", "metricas.csv", "curva_perda.csv"):
        (d / f).unlink(missing_ok=True)

    import aeon
    import tensorflow as tf

    gpus = tf.config.list_physical_devices("GPU")
    for g in gpus:
        try:
            tf.config.experimental.set_memory_growth(g, True)
        except Exception:  # noqa: BLE001
            pass
    gpu_name = ""
    if gpus:
        try:
            gpu_name = tf.config.experimental.get_device_details(gpus[0]).get("device_name", "")
        except Exception:  # noqa: BLE001
            pass
    status = {
        "timestamp_inicio": datetime.now().isoformat(timespec="seconds"),
        "dataset": dataset, "model": model, "status": "em_andamento",
        "seed": cfg["seed"], "n_epochs": cfg["epochs"],
        "k_grid": f"{cfg['k_min']}:{cfg['k_max']}:{cfg['k_step']}",
        "host": socket.gethostname(), "device": "GPU" if gpus else "CPU",
        "gpu_name": gpu_name, "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES", ""),
        "python": platform.python_version(), "tensorflow": tf.__version__,
        "aeon": aeon.__version__,
    }
    try:
        t0 = time.perf_counter()
        data = load_dataset(dataset, Path(cfg["data_dir"]))
        X, y, n_train = data["X"], data["y"], data["n_train"]
        n_cases, _, n_tp = X.shape
        n_classes = len(np.unique(y))
        status.update(load_time_s=round(time.perf_counter() - t0, 3), n_cases=n_cases,
                      n_train=n_train, n_test=n_cases - n_train, n_timepoints=n_tp,
                      n_classes=n_classes, preprocessing=data["preprocessing"])
        print(f"{dataset}/{model}: n={n_cases} m={n_tp} classes={n_classes} "
              f"[{status['device']} {gpu_name}]", flush=True)

        latent, info, curve = train_and_extract(model, X, cfg["epochs"], cfg["seed"])
        status.update(info)
        np.save(d / "latente.npy", latent)
        _write_csv(d / "curva_perda.csv",
                   ["dataset", "model", "epoch", "loss", "learning_rate", "epoch_time_s"],
                   [{"dataset": dataset, "model": model, **r} for r in curve])
        print(f"  treino {info['train_time_s']:.1f}s, {info['epochs_run']} epocas, "
              f"perda min {info['recon_loss_min']:.4g} | {info['encoder_check']}", flush=True)

        base = {"dataset": dataset, "model": model, "seed": cfg["seed"],
                "n_epochs": cfg["epochs"], "n_cases": n_cases, "n_timepoints": n_tp,
                "n_classes": n_classes, "preprocessing": data["preprocessing"]}
        rows, notes, n_k = [], [], 0
        t0 = time.perf_counter()
        for km in cfg["k_modes"]:
            ks = ks_for(km, n_cases, n_classes, cfg)
            r, nt = k_search(latent, y, n_train, ks, km, cfg["seed"], base,
                             info["recon_loss_min"], info["recon_mse"])
            rows += r
            notes += [f"[{km}] {s}" for s in nt]
            n_k += len(ks)
        status["ksearch_time_s"] = round(time.perf_counter() - t0, 3)
        status["n_k"] = n_k
        _write_csv(d / "metricas.csv", LONG_COLS, rows)
        status.update(status="ok", message=" | ".join(notes)[:4000])
        print(f"  busca de K: {n_k} valores em {status['ksearch_time_s']:.1f}s", flush=True)
        rc = 0
    except Exception as e:  # noqa: BLE001
        tb = traceback.format_exc(limit=4)
        status.update(status="erro",
                      message=" ".join(f"{type(e).__name__}: {e} || {tb}".split())[:4000])
        print(f"  ERRO {type(e).__name__}: {e}", flush=True)
        rc = 1
    status["timestamp_fim"] = datetime.now().isoformat(timespec="seconds")
    _write_status(d, status)
    return rc


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--config", required=True)
    p.add_argument("--dataset", required=True)
    p.add_argument("--model", required=True, choices=MODELS)
    a = p.parse_args(argv)
    cfg = json.loads(Path(a.config).read_text(encoding="utf-8"))
    import keras

    keras.utils.disable_interactive_logging()
    return run_pair(cfg, a.dataset, a.model)


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile aeon_dl_agendador.py
"""Agenda os pares (conjunto, modelo) em subprocessos, um por GPU, e consolida os CSVs.

    python aeon_dl_agendador.py --config <out>/config.json            # roda + consolida
    python aeon_dl_agendador.py --config <out>/config.json --so-consolidar

- Retomavel: pula pares com status "ok" (e, se repetir_falhas=False, tambem os que falharam).
- Um subprocesso por par: se faltar memoria ou o processo travar, so aquele par falha,
  e a falha fica registrada (status "erro", "erro_processo" ou "timeout").
- Ordem: conjuntos menores primeiro (n_casos x comprimento).
- Paralelismo: um "slot" por GPU listada em config["gpus"] (x jobs_por_gpu).
"""

from __future__ import annotations

import argparse
import json
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
import aeon_dl_par as P  # noqa: E402

EXEC_COLS = [
    "timestamp_inicio", "timestamp_fim", "dataset", "model", "status", "seed", "n_epochs",
    "k_grid", "host", "device", "gpu_name", "cuda_visible_devices", "python", "tensorflow",
    "aeon", "n_cases", "n_train", "n_test", "n_timepoints", "n_classes", "preprocessing",
    "n_params", "latent_dim", "load_time_s", "train_time_s", "epochs_run", "best_epoch",
    "first_epoch_s", "epoch_median_s", "latent_time_s", "ksearch_time_s", "n_k",
    "recon_loss_min", "recon_loss_last", "recon_mse", "lr_final", "gpu_mem_peak_mb",
    "wall_time_s", "encoder_check", "message",
]


def log(msg: str, out: Path | None = None):
    line = f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}"
    print(line, flush=True)
    if out is not None:
        with open(out / "agendador.log", "a", encoding="utf-8") as f:
            f.write(line + "\n")


def read_status(out: Path, dataset: str, model: str) -> dict | None:
    f = P.pair_dir(out, dataset, model) / "status.json"
    if not f.exists():
        return None
    try:
        return json.loads(f.read_text(encoding="utf-8"))
    except Exception:  # noqa: BLE001
        return None


def same_config(st: dict, cfg: dict) -> bool:
    return (str(st.get("seed")) == str(cfg["seed"])
            and str(st.get("n_epochs")) == str(cfg["epochs"])
            and st.get("k_grid") == f"{cfg['k_min']}:{cfg['k_max']}:{cfg['k_step']}")


def dataset_sizes(cfg: dict, out: Path) -> dict[str, tuple[int, int]]:
    """Carrega (e guarda em cache) todos os conjuntos para ordenar por tamanho."""
    shapes = {}
    for i, name in enumerate(cfg["datasets"], 1):
        try:
            d = P.load_dataset(name, Path(cfg["data_dir"]))
            shapes[name] = (int(d["X"].shape[0]), int(d["X"].shape[2]))
        except Exception as e:  # noqa: BLE001
            log(f"erro ao carregar {name}: {type(e).__name__}: {e}", out)
            for model in cfg["models"]:
                d = P.pair_dir(out, name, model)
                d.mkdir(parents=True, exist_ok=True)
                P._write_status(d, {"dataset": name, "model": model, "status": "erro_carga",
                                    "seed": cfg["seed"], "n_epochs": cfg["epochs"],
                                    "k_grid": f"{cfg['k_min']}:{cfg['k_max']}:{cfg['k_step']}",
                                    "timestamp_fim": datetime.now().isoformat(timespec="seconds"),
                                    "message": f"{type(e).__name__}: {e}"[:2000]})
        if i % 16 == 0:
            log(f"dados: {i}/{len(cfg['datasets'])} conjuntos prontos", out)
    return shapes


def run(cfg: dict) -> None:
    out = Path(cfg["out"])
    (out / "logs").mkdir(parents=True, exist_ok=True)
    t_start = time.time()
    log(f"inicio | {len(cfg['datasets'])} conjuntos x {len(cfg['models'])} modelos | "
        f"epocas={cfg['epochs']} seed={cfg['seed']} K={cfg['k_min']}:{cfg['k_max']}:{cfg['k_step']} "
        f"k_modes={cfg['k_modes']} gpus={cfg['gpus']}", out)
    shapes = dataset_sizes(cfg, out)
    order = sorted(shapes, key=lambda n: (shapes[n][0] * shapes[n][1], n))
    if cfg.get("max_size"):
        order = [n for n in order if shapes[n][0] * shapes[n][1] <= cfg["max_size"]]

    queue = []
    for name in order:
        for model in cfg["models"]:
            st = read_status(out, name, model)
            if st and same_config(st, cfg):
                if st.get("status") == "ok":
                    continue
                if not cfg.get("repetir_falhas", False) and st.get("status") != "em_andamento":
                    continue
            queue.append((name, model))
    log(f"fila: {len(queue)} pares pendentes", out)

    slots = [str(g) for g in cfg["gpus"]] * int(cfg.get("jobs_por_gpu", 1)) or ["cpu"]
    running: dict[str, tuple] = {}  # slot -> (proc, name, model, t0, logfile)
    timeout = float(cfg.get("timeout_par_horas") or 0) * 3600
    deadline = float(cfg.get("parar_apos_horas") or 0) * 3600
    last_consol = time.time()

    def launch(slot, name, model):
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = "" if (slot == "cpu" or model in cfg.get("modelos_cpu", [])) else slot
        env["TF_CPP_MIN_LOG_LEVEL"] = "2"
        lf = open(out / "logs" / f"{model}__{name}.log", "w", encoding="utf-8")
        cmd = [sys.executable, "-u", str(HERE / "aeon_dl_par.py"), "--config",
               str(out / "config.json"), "--dataset", name, "--model", model]
        proc = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT, env=env)
        running[slot] = (proc, name, model, time.time(), lf)
        log(f"[slot {slot}] inicia {name}/{model}", out)

    def finish(slot, rc, reason=""):
        proc, name, model, t0, lf = running.pop(slot)
        lf.close()
        wall = round(time.time() - t0, 1)
        st = read_status(out, name, model)
        d = P.pair_dir(out, name, model)
        if st is None or st.get("status") == "em_andamento" or reason:
            # O filho morreu sem gravar status (ex.: SIGKILL por falta de memoria) ou estourou o tempo.
            msg = reason or (f"subprocesso terminou com codigo {rc} sem registrar resultado"
                             + (" (SIGKILL: provavel falta de memoria)" if rc in (-9, 137) else ""))
            tail = ""
            try:
                tail = " ".join((out / "logs" / f"{model}__{name}.log").read_text(
                    encoding="utf-8", errors="replace").split()[-60:])
            except Exception:  # noqa: BLE001
                pass
            st = {"dataset": name, "model": model,
                  "status": "timeout" if reason else "erro_processo",
                  "seed": cfg["seed"], "n_epochs": cfg["epochs"],
                  "k_grid": f"{cfg['k_min']}:{cfg['k_max']}:{cfg['k_step']}",
                  "timestamp_fim": datetime.now().isoformat(timespec="seconds"),
                  "message": f"{msg} || log: {tail}"[:4000], **(shapes_info(name))}
        st["wall_time_s"] = wall
        d.mkdir(parents=True, exist_ok=True)
        P._write_status(d, st)
        log(f"[slot {slot}] fim {name}/{model}: {st['status']} ({wall:.0f}s)", out)
        for tmp in Path("/dev/shm").glob("aeon_ckpt_*") if Path("/dev/shm").is_dir() else []:
            if time.time() - tmp.stat().st_mtime > 60 and not running:
                shutil.rmtree(tmp, ignore_errors=True)

    def shapes_info(name):
        n, m = shapes.get(name, ("", ""))
        return {"n_cases": n, "n_timepoints": m}

    try:
        while queue or running:
            for slot in list(running):
                proc, name, model, t0, _ = running[slot]
                rc = proc.poll()
                if rc is not None:
                    finish(slot, rc)
                elif timeout and time.time() - t0 > timeout:
                    proc.kill()
                    proc.wait()
                    finish(slot, -9, f"timeout: passou de {cfg['timeout_par_horas']} h")
            stop = deadline and time.time() - t_start > deadline
            for slot in slots:
                if slot not in running and queue and not stop:
                    launch(slot, *queue.pop(0))
            if stop and queue:
                log(f"prazo de {cfg['parar_apos_horas']} h atingido; {len(queue)} pares ficam "
                    "pendentes (rode de novo para retomar)", out)
                queue.clear()
            if time.time() - last_consol > 600:
                consolidate(cfg)
                last_consol = time.time()
            time.sleep(2)
    finally:
        for slot in list(running):
            running[slot][0].kill()
            finish(slot, -9, "agendador interrompido")
        consolidate(cfg)
        log(f"fim | {time.time() - t_start:.0f}s", out)


# ---------------------------------------------------------------- consolidacao
def consolidate(cfg: dict) -> dict:
    """Junta as pastas dos pares nos CSVs finais (reescritos do zero a cada chamada)."""
    import pandas as pd

    out = Path(cfg["out"])
    base = out / "pares"
    status, longs, curves = [], [], []
    if base.exists():
        for f in sorted(base.glob("*/*/status.json")):
            try:
                st = json.loads(f.read_text(encoding="utf-8"))
            except Exception:  # noqa: BLE001
                continue
            status.append(st)
            if st.get("status") == "ok":
                if (f.parent / "metricas.csv").exists():
                    longs.append(pd.read_csv(f.parent / "metricas.csv"))
                if (f.parent / "curva_perda.csv").exists():
                    curves.append(pd.read_csv(f.parent / "curva_perda.csv"))

    ex = pd.DataFrame(status)
    for c in EXEC_COLS:
        if c not in ex.columns:
            ex[c] = ""
    ex = ex[EXEC_COLS + [c for c in ex.columns if c not in EXEC_COLS]]
    if not ex.empty:
        ex = ex.sort_values(["dataset", "model"])
    ex.to_csv(out / "execucoes.csv", index=False)
    ex[ex["status"] != "ok"].to_csv(out / "falhas.csv", index=False)

    lg = pd.concat(longs, ignore_index=True) if longs else pd.DataFrame(columns=P.LONG_COLS)
    lg.to_csv(out / "resultados_longo.csv", index=False)
    (pd.concat(curves, ignore_index=True) if curves
     else pd.DataFrame(columns=["dataset", "model", "epoch", "loss", "learning_rate",
                                "epoch_time_s"])).to_csv(out / "curvas_perda.csv", index=False)

    keys = ["dataset", "model", "seed", "n_epochs", "n_cases", "n_timepoints", "n_classes",
            "preprocessing", "k_mode", "k"]
    if lg.empty:
        wide = pd.DataFrame(columns=keys + P.METRICS)
    else:
        wide = (lg.pivot_table(index=keys, columns="metric", values="value", aggfunc="first",
                               dropna=False).reset_index())
        wide.columns.name = None
        wide = wide[keys + [m for m in P.METRICS if m in wide.columns]]
        wide = wide.dropna(subset=["n_cases"]).sort_values(["dataset", "model", "k_mode", "k"])
    wide.to_csv(out / "resultados_largo.csv", index=False)

    # K escolhido pela silhueta (maior; empate -> menor K), entre solucoes nao degeneradas.
    sel = []
    g = wide[wide["k_mode"] == "grid"] if not wide.empty else wide
    for (ds, mdl), grp in (g.groupby(["dataset", "model"]) if not g.empty else []):
        ok = grp.dropna(subset=["silhouette"])
        if ok.empty:
            continue
        b = ok.sort_values(["silhouette", "k"], ascending=[False, True]).iloc[0]
        row = b.rename({"k": "k_selected"}).drop(labels=["k_mode"]).to_dict()
        orc = grp.dropna(subset=["ari"]).sort_values(["ari", "k"], ascending=[False, True])
        if not orc.empty:
            row["k_oracle_ari"], row["ari_oracle"] = orc.iloc[0]["k"], orc.iloc[0]["ari"]
        nc = wide[(wide.dataset == ds) & (wide.model == mdl) & (wide.k_mode == "n_classes")]
        for m in ("ri", "ari", "nmi", "ami"):
            row[f"{m}_k_n_classes"] = nc.iloc[0][m] if not nc.empty else ""
        sel.append(row)
    pd.DataFrame(sel).to_csv(out / "selecao_k_silhueta.csv", index=False)

    # Situacao por conjunto
    ds_rows = []
    if not ex.empty:
        for ds, grp in ex.groupby("dataset"):
            r = {"dataset": ds}
            for c in ("n_cases", "n_timepoints", "n_classes", "preprocessing"):
                v = grp[c].replace("", pd.NA).dropna()
                r[c] = v.iloc[0] if not v.empty else ""
            r["modelos_ok"] = int((grp.status == "ok").sum())
            r["modelos_falha"] = int((grp.status != "ok").sum())
            r["modelos_pendentes"] = len(cfg["models"]) - len(grp)
            r["falhas"] = ";".join(f"{m}:{s}" for m, s in zip(grp.model, grp.status) if s != "ok")
            ds_rows.append(r)
    pd.DataFrame(ds_rows).to_csv(out / "status_conjuntos.csv", index=False)

    resumo = {"pares_ok": int((ex.status == "ok").sum()) if not ex.empty else 0,
              "pares_falha": int((ex.status != "ok").sum()) if not ex.empty else 0,
              "pares_total": len(cfg["datasets"]) * len(cfg["models"]),
              "conjuntos_com_resultado": int(lg["dataset"].nunique()) if not lg.empty else 0,
              "linhas_resultados_longo": len(lg)}
    (out / "progresso.json").write_text(json.dumps(resumo, indent=1), encoding="utf-8")
    return resumo


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--config", required=True)
    p.add_argument("--so-consolidar", action="store_true")
    a = p.parse_args(argv)
    cfg = json.loads(Path(a.config).read_text(encoding="utf-8"))
    if a.so_consolidar:
        print(json.dumps(consolidate(cfg), indent=1))
        return 0
    run(cfg)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

## 4. Conferência do ambiente

In [ ]:
import importlib, platform
print("python", platform.python_version())
for m in ["aeon", "tensorflow", "keras", "sklearn", "numpy", "pandas"]:
    try:
        print(f"{m:12s}", importlib.import_module(m).__version__)
    except Exception as e:
        print(f"{m:12s} FALTANDO: {e}")
r = subprocess.run([sys.executable, "-c",
    "import os; os.environ['TF_CPP_MIN_LOG_LEVEL']='2'; import tensorflow as tf; "
    "print('GPUs vistas pelo TensorFlow:', tf.config.list_physical_devices('GPU'))"],
    capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-2000:])

## 5. Grava a configuração

In [ ]:
from aeon.datasets.tsc_datasets import univariate

cfg = dict(out=str(OUT), data_dir=str(DATA_DIR), epochs=EPOCHS, seed=SEED,
           k_min=K_MIN, k_max=K_MAX, k_step=K_STEP, k_modes=K_MODES,
           models=MODELS, modelos_cpu=MODELOS_CPU,
           datasets=DATASETS or sorted(univariate),
           gpus=GPUS, jobs_por_gpu=JOBS_POR_GPU, timeout_par_horas=TIMEOUT_PAR_HORAS,
           parar_apos_horas=PARAR_APOS_HORAS, repetir_falhas=REPETIR_FALHAS)
OUT.mkdir(parents=True, exist_ok=True)
CFG = OUT / "config.json"
CFG.write_text(json.dumps(cfg, indent=1))
print(f"{len(cfg['datasets'])} conjuntos x {len(MODELS)} modelos -> {CFG}")

## 6. Executa

In [ ]:
cmd = [sys.executable, "-u", str(Path("aeon_dl_agendador.py").resolve()), "--config", str(CFG)]
if EM_SEGUNDO_PLANO:
    # Destacado do kernel: continua rodando se o notebook for fechado.
    lf = open(OUT / "agendador_stdout.log", "a")
    p = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT, start_new_session=True)
    (OUT / "agendador.pid").write_text(str(p.pid))
    print("agendador rodando em segundo plano, PID", p.pid, "| log:", OUT / "agendador.log")
else:
    rc = subprocess.run(cmd).returncode
    print("código de saída:", rc)

## 7. Acompanhamento (rode quantas vezes quiser)

In [ ]:
import subprocess, json
subprocess.run([sys.executable, "aeon_dl_agendador.py", "--config", str(CFG), "--so-consolidar"],
               capture_output=True)
print(json.loads((OUT / "progresso.json").read_text()))
log = OUT / "agendador.log"
if log.exists():
    print("".join(log.read_text().splitlines(True)[-15:]))

In [ ]:
# Para parar o agendador em segundo plano (os pares concluídos ficam salvos):
# import os, signal; os.killpg(int((OUT / "agendador.pid").read_text()), signal.SIGTERM)

## 8. Resultados

In [ ]:
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
for f in ["resultados_longo.csv", "resultados_largo.csv", "selecao_k_silhueta.csv",
          "execucoes.csv", "curvas_perda.csv", "status_conjuntos.csv", "falhas.csv"]:
    p = OUT / f
    df = pd.read_csv(p) if p.exists() and p.stat().st_size else pd.DataFrame()
    print(f"\n### {f}: {len(df)} linhas")
    display(df.head(5))

## 9. Arquivos de saída e dicionário de colunas

Tudo em `OUT/`. Os CSVs da raiz são reconstruídos a partir de `pares/` a cada consolidação.

| Arquivo | Conteúdo |
|---|---|
| `resultados_longo.csv` | **Principal.** Uma linha por (conjunto, modelo, K, métrica): `dataset, model, k, k_mode, metric, value, seed, n_epochs, n_cases, n_timepoints, n_classes, preprocessing` |
| `resultados_largo.csv` | O mesmo, uma linha por (conjunto, modelo, k_mode, K), métricas em colunas |
| `selecao_k_silhueta.csv` | Por (conjunto, modelo): K escolhido pela maior silhueta (empate: menor K) e todas as métricas nesse K; também `ri/ari/nmi/ami_k_n_classes` e o melhor K pelo ARI (`k_oracle_ari`, `ari_oracle`, usa rótulos: só limite otimista) |
| `execucoes.csv` | Uma linha por par: status, máquina, GPU, versões, tamanhos, parâmetros da rede, tempos, épocas, perdas, memória de GPU, mensagem de erro |
| `curvas_perda.csv` | Perda, taxa de aprendizado e tempo de cada época de cada treino |
| `status_conjuntos.csv` | Por conjunto: quantos modelos ok, com falha e pendentes |
| `falhas.csv` | Só os pares que falharam, com a mensagem e o fim do log |
| `pares/<modelo>/<conjunto>/` | `metricas.csv`, `curva_perda.csv`, `latente.npy`, `status.json` de cada par |
| `logs/` | saída de cada subprocesso; `agendador.log` |

**Métricas por K** (`metric` no CSV longo):

| Métrica | Significado |
|---|---|
| `silhouette`, `davies_bouldin`, `calinski_harabasz` | Índices internos no espaço latente (sem rótulos). Silhueta e CH: maior é melhor; DB: menor é melhor |
| `ri`, `ari`, `nmi`, `ami` | Externos, contra as classes, no conjunto inteiro (treino+teste) |
| `ri_train`, `ari_train`, `nmi_train` / `ri_test`, `ari_test`, `nmi_test` | O **mesmo** agrupamento (ajustado no conjunto unido) avaliado só nas séries que vinham do treino / do teste no split original do UCR |
| `n_clusters_produced` | Grupos que o k-means de fato produziu |
| `n_empty_components` | K − grupos produzidos |
| `degenerate` | 1 se produziu menos de 2 grupos (silhueta/DB/CH ficam NaN) |
| `cluster_size_min`, `cluster_size_max` | Tamanho do menor e do maior grupo |
| `kmeans_inertia`, `kmeans_n_iter` | Inércia final e iterações do k-means |
| `recon_loss` | Menor perda de treino (o aeon recarrega esse modelo). No AEDCNN inclui a penalidade L2 |
| `recon_mse` | MSE pura de reconstrução do modelo final em todo o conjunto (comparável entre modelos) |
| `elbo` | Vazio: os autoencoders do aeon não são variacionais |

**`execucoes.csv`, colunas de treino:** `n_params` (parâmetros da rede), `latent_dim`,
`train_time_s`, `epochs_run`, `best_epoch` (época de menor perda), `first_epoch_s` (inclui
compilação), `epoch_median_s`, `latent_time_s`, `ksearch_time_s`, `n_k`, `recon_loss_min`,
`recon_loss_last`, `recon_mse`, `lr_final`, `gpu_mem_peak_mb`, `wall_time_s` (tempo total do
subprocesso), `encoder_check` (confere que `layers[1]` é o codificador), `status`
(`ok`, `erro`, `erro_processo`, `timeout`, `erro_carga`) e `message`.